# 25 — `integrate_with_corrections`: the one-call recipe

When you want the full physics — polarization + solid angle + residual
correction map + Q-uniform binning + variance propagation — compose them
yourself OR call `integrate_with_corrections` which wires the standard
chain in the canonical order.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

In [ ]:
from midas_integrate_v2 import IntegrationSpec
from midas_integrate_v2.corrections import (
    integrate_with_corrections,
    PolarizationCorrection, SolidAngleCorrection,
    RBFResidualCorrection, IdentityResidualCorrection,
)

t = lambda x: torch.as_tensor(x, dtype=torch.float64)
spec = IntegrationSpec()
spec.NrPixelsY = 2048; spec.NrPixelsZ = 2048
spec.pxY = 200.0; spec.pxZ = 200.0
spec.Lsd = t(940000.0); spec.BC_y = t(1024.0); spec.BC_z = t(994.0)
spec.tx = t(0.0); spec.ty = t(0.0); spec.tz = t(0.0)
spec.Wavelength = t(0.189714); spec.Parallax = t(0.0); spec.RhoD = 1024.0
spec.RMin = 100.0; spec.RMax = 1800.0; spec.RBinSize = 0.5
spec.EtaMin = -180.0; spec.EtaMax = 180.0; spec.EtaBinSize = 1.0

# Stand-in image
img = torch.rand(spec.NrPixelsZ, spec.NrPixelsY, dtype=torch.float64) * 1000.0

result = integrate_with_corrections(
    img, spec,
    polarization=PolarizationCorrection(P_horizontal=0.99),
    solid_angle=SolidAngleCorrection(),
    residual_corr=IdentityResidualCorrection(),    # or RBFResidualCorrection(...)
    # apply_compton=False  (default; turn on for PDF)
)
print('result keys:', list(result.__dataclass_fields__) if hasattr(result, '__dataclass_fields__') else dir(result))

## Variance-aware variants

`integrate_hard_with_variance`, `integrate_subpixel_with_variance`,
`integrate_polygon_with_variance` propagate per-pixel σ² through the
binning. Required for honest Rietveld weights and PDF error bars.

Use:
```python
from midas_integrate_v2 import (
    integrate_hard_with_variance,
    integrate_subpixel_with_variance,
    integrate_polygon_with_variance,
)
profile, sigma = integrate_polygon_with_variance(img, geom, pixel_var=...)
```